# 表面质量 — mesh 可视化 + 诊断

散点图看不出预测表面好不好，这份 notebook 把点云转成能看的 mesh，
再加上 mesh 看不见的两个诊断图和一组客观数字。

**用途**：现在渲染一版存档 → 改训练代码 → 再渲染一版 → 前后对比。

---

## 三件套，缺一不可

| | 回答什么问题 | 为什么单独它不够 |
|---|---|---|
| **① mesh 对比** | 整体形状对不对、表面平不平 | 会把点的疏密和散布抹掉 |
| **② 诊断彩色点云** | 点是散在表面两侧、还是扎堆 | 看不出整体形状 |
| **③ 数字** | 到底有没有变好 | 前两个都是肉眼判断，会骗人 |

## ⚠️ 两条硬规则

**1. 这些 mesh 只能用来「看」，不能用来算指标。**
重建会把形状整体外扩：实测原始点到重建表面的距离中位 **5.3mm**、p95 12.0mm，
和模型自己的 CD_t（约 7mm）是同一量级。所有 Chamfer/DCD 必须继续用原始点云算
（`msn_skullfix.calc_cd` / `calc_dcd`）。

**2. 对比不同版本时，重建参数必须完全一致。**
`mesh_viz.RECON` 是模块级常量，不要为了「让这张图好看」去调。
每个旋钮都会改变表面看起来多平滑，改了就分不清是模型进步还是参数调松。

> **一个反直觉但实测的结论**：平滑加得越多，模型之间的差异反而**越明显**。
> 低平滑时 GT 和预测都渲染成同一堆小球，重建自身的纹理淹没了真实差异；
> 加大平滑把这层纹理滤掉，剩下的低频起伏才是模型真正的差距。

## 1. 配置

`MODELS` 里每加一行，下面所有图表就多一列 —— 改完训练代码后，
把新 checkpoint 追加进去就能和现在这版并排比。

In [ ]:
import os, sys, json

# 仓库根目录：从当前工作目录向上找，直到看见 src/models/
REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"没找到仓库根目录，当前在 {os.getcwd()}"

CACHE = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")

# 要对比的模型：(显示名, run 目录名)。改完训练代码后在这里追加新的 run-name。
MODELS = [
    ("baseline_es20", "baseline_es20"),
    # ("dcd_w5",      "dcd_w5"),        # 例：调完 dcd_weight 之后
]

SKULL   = None        # None = 用验证集第一颗；也可以写 "083" 指定
N_STATS = 8           # 数字统计用多少颗验证集颅骨（渲染只用 1 颗）
DEVICE  = "/GPU:0"    # 显存被别的 kernel 占住时改 "/CPU:0"

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
sys.path.insert(0, os.path.join(REPO, "src", "models"))
sys.path.insert(0, os.path.join(REPO, "src", "eval"))
print("REPO =", REPO)

## 2. 载入数据与模型，跑推理

In [ ]:
import numpy as np
import tensorflow as tf
for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

import msn_skullfix as msn
import mesh_viz as mv

data = np.load(CACHE)
ids, inputs, gt, scale_mm = data["ids"], data["inputs"], data["gt"], data["scale_mm"]
text_feat = np.load(os.path.join(REPO, "data", "cache", "bert_skull.npy"))

# 所有模型共用同一份验证集划分（各 run 的 run.json 里都记着，理应一致）
meta0 = json.load(open(os.path.join(REPO, "experiments", MODELS[0][1], "run.json")))
val_ids = meta0["val_ids"]
sel = [i for i, s in enumerate(ids) if s in set(val_ids)][:N_STATS]
k_show = sel[0] if SKULL is None else [i for i, s in enumerate(ids) if s == SKULL][0]
print(f"渲染 skull_{ids[k_show]} | 统计用 {len(sel)} 颗验证集颅骨")

preds = {}          # name -> {idx: (6144,3)}
with tf.device(DEVICE):
    for name, run in MODELS:
        model = msn.build_model(msn.MSNConfig.paper())
        model.load_weights(os.path.join(REPO, "experiments", run, "best.h5"))
        preds[name] = {i: model([inputs[i][None], text_feat[None]],
                                training=False).numpy()[0] for i in sel}
        del model
        tf.keras.backend.clear_session()
        print(f"  {name}: 推理完成")

## 3. 数字先行

先看数字再看图 —— 图会骗人，数字不会。

- **扎堆<2mm**：最近邻不到 2mm 的点占比。这些点几乎重合、覆盖不到新表面，等于浪费。
  GT 是最远点采样出来的，这一项**恒为 0.0%**，所以它是个干净的参照。
- **间距CV**：最近邻距离的变异系数，衡量疏密是否均匀。越接近 GT 越好。
- **|偏差|**：预测点到 GT 局部表面的距离。衡量表面位置准不准。
- **外侧占比**：偏差为正的比例。接近 50% 说明是对称散布（没有系统性胀大或缩小）。

In [ ]:
rows = []
for name, _ in MODELS:
    per = [mv.surface_stats(preds[name][i], gt[i], float(scale_mm[i])) for i in sel]
    rows.append((name, {k: float(np.mean([p[k] for p in per])) for k in per[0]}))
print(mv.format_stats(rows))

## 4. ① mesh 对比

GT 放在最左边当参照。看的是**低频起伏**：GT 应该圆润连续，
预测如果有波浪状凹凸，就是表面精度不够。

（第一次跑要几秒，每个点云约 1 秒。）

In [ ]:
s_show = float(scale_mm[k_show])
items = [(mv.pc_to_mesh(gt[k_show], s_show), "ground truth")]
items += [(mv.pc_to_mesh(preds[n][k_show], s_show), n) for n, _ in MODELS]
mv.fig_meshes(items, f"skull_{ids[k_show]} — 表面对比（重建参数已锁定）").show()

## 5. ② 诊断：mesh 看不见的两件事

- **左图 有符号偏差**：红=在 GT 表面外侧，蓝=内侧，白=贴合。
  红蓝均匀混杂 = 点散布在表面两侧（就是"有的高有的低"）；
  大片同色 = 该区域整体鼓出或塌陷。
- **右图 最近邻间距**：暗色=和邻点挤在一起。暗点越多，浪费的点越多。

In [ ]:
for name, _ in MODELS:
    mv.fig_diagnostic(preds[name][k_show], gt[k_show], s_show,
                      f"skull_{ids[k_show]} — {name}").show()

## 6. 存档

改训练代码之前先跑一次这里，把当前状态存下来，改完之后才有得比。

In [ ]:
import pandas as pd
OUT = os.path.join(REPO, "experiments_log", "surface_quality.csv")
df = pd.DataFrame([{"model": n, **s} for n, s in rows])
if os.path.exists(OUT):                      # 追加，保留历史版本
    df = pd.concat([pd.read_csv(OUT), df]).drop_duplicates("model", keep="last")
os.makedirs(os.path.dirname(OUT), exist_ok=True)
df.to_csv(OUT, index=False)
print(f"-> {OUT}")
df